# NCERT Class 8 Science — AI Tutor (RAG Pipeline)

This notebook walks through the full retrieval-augmented generation (RAG) pipeline for the Class 8 Science tutor:

1. **Ingestion** — extract text from NCERT PDFs, clean it, and chunk it into a JSONL corpus  
2. **Indexing** — embed every chunk with a local sentence-transformer model and store vectors in FAISS  
3. **Retrieval & Generation** — embed a question, find the closest chunks, and use Gemini Flash to produce a grounded answer  
4. **Evaluation** — score the system on 20 test questions using BLEU and ROUGE-L  

Drop your `ch01.pdf … ch13.pdf` files into `data/raw/` before running. If the folder is empty the pipeline falls back to a small built-in demo corpus so every cell still runs.

## 0. Setup

Add the project root to `sys.path` so we can import from `src/` regardless of where Jupyter was launched.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
# Walk up until we find requirements.txt (the project root marker)
for parent in [project_root] + list(project_root.parents):
    if (parent / 'requirements.txt').exists():
        project_root = parent
        break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print('Project root:', project_root)

## Stage 1 — Text Ingestion

`src/ingest.py` does three things:

- **Extract** raw text from each chapter PDF using `pdfplumber`
- **Clean** the text: strip bare page numbers, running headers/footers, NCERT rationalisation notices, and collapse extra whitespace
- **Chunk** the cleaned text into ~250-word windows with a 50-word overlap so that no fact gets split across a boundary

Each chunk is saved as one JSON line in `data/processed/class8_science.jsonl`:

```json
{"id": "ch01_0", "content": "...", "meta": {"chapter": "ch01", "chapter_title": "Crop Production and Management"}}
```

If `data/raw/` is empty the function produces a small demo corpus (one chunk per chapter) so the rest of the notebook still runs end-to-end.

In [ ]:
from src.ingest import ingest_documents

documents = ingest_documents()
print(f'Corpus size: {len(documents)} chunks')
print('Sample chunk:')
import json
print(json.dumps(documents[0], indent=2, ensure_ascii=False))

## Stage 2 — Vector Indexing

`src/index_build.py` turns the text corpus into a searchable vector index:

- **Embedding model**: `all-MiniLM-L6-v2` (384 dimensions, runs locally, no API cost)
- **Normalisation**: vectors are L2-normalised so that inner-product search equals cosine similarity
- **Index type**: `faiss.IndexFlatIP` — exact, brute-force inner-product search; fast enough for a few thousand chunks

Two files are written to `data/processed/`:
- `faiss.index` — the binary FAISS index
- `documents.pkl` — the ordered list of doc dicts, so row *N* in the index maps back to doc *N*

In [ ]:
from src.index_build import build_index
import faiss
from pathlib import Path

build_index()

index = faiss.read_index(str(project_root / 'data' / 'processed' / 'faiss.index'))
print(f'Index vectors: {index.ntotal}  |  Dimension: {index.d}')

## Stage 3 — Retrieval and Answer Generation

`src/rag.py` ties retrieval and generation together.

### `retrieve(query, k=4)`
Embeds the query with the same `all-MiniLM-L6-v2` model, runs an inner-product search in FAISS, and returns the top-*k* `(doc, score)` pairs.

### `answer(query, k=4, threshold=0.3)`
1. Calls `retrieve()` to get the closest chunks.
2. **Fallback gate**: if the top cosine score is below `threshold` (0.3) the question is considered out-of-syllabus and a canned fallback is returned without calling the Gemini API.
3. Otherwise, formats the chunks as a context block and calls **Gemini Flash** (free tier) with a system prompt that restricts answers to the provided context, mandates grade-appropriate language, and always cites the source chapter.
4. Returns `(answer_text, list_of_source_chapter_titles)`.

The Gemini model ID is read from `GEMINI_MODEL` in `.env` — update it to match the current free Flash model shown in your AI Studio dashboard.

In [ ]:
from src.rag import retrieve, answer

# Show raw retrieval scores for a sample query
query = 'What causes friction between surfaces?'
hits = retrieve(query, k=4)
print(f'Top {len(hits)} chunks for: "{query}"\n')
for doc, score in hits:
    print(f'  [{score:.3f}]  {doc["meta"]["chapter_title"]} — {doc["content"][:80]}...')

In [ ]:
import time

sample_questions = [
    'What is friction useful for?',
    'How do microorganisms help in making curd?',
    'Why do plants need sunlight?',
    'What is the capital of France?',   # out-of-syllabus — should trigger fallback
]

for q in sample_questions:
    ans, sources = answer(q)
    print(f'Q: {q}')
    print(f'A: {ans}')
    if sources:
        print(f'Sources: {", ".join(sources)}')
    print('---')
    time.sleep(1)  # respect the free-tier rate limit

## Stage 4 — Evaluation

We measure answer quality on 20 questions from `eval/test_questions.json` — 16 are from the Class 8 syllabus and 4 are deliberately out-of-syllabus to test the fallback.

**Metrics:**
- **BLEU** (with `SmoothingFunction.method1`) — n-gram precision between generated and reference answers. Good for exact-wording overlap.
- **ROUGE-L F-measure** — longest common subsequence ratio. Better at capturing paraphrase.

Neither metric is perfect for free-form answers, so the CSV also has a blank `human_review` column for manual assessment.

A 0.8-second sleep between calls keeps requests inside the free-tier rate limit (~10–15 req/min).

In [ ]:
import json
import time
import pandas as pd
from pathlib import Path
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from src.rag import answer

questions_path = project_root / 'eval' / 'test_questions.json'
output_path = project_root / 'eval' / 'evaluation.csv'

with questions_path.open('r', encoding='utf-8') as f:
    questions = json.load(f)

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
smoothie = SmoothingFunction().method1
rows = []

for item in questions:
    q, ref = item['question'], item['reference']
    gen, sources = answer(q)
    bleu = sentence_bleu([ref.split()], gen.split(), smoothing_function=smoothie)
    rougeL = scorer.score(ref, gen)['rougeL'].fmeasure
    rows.append({
        'question': q,
        'generated': gen,
        'reference': ref,
        'sources': ', '.join(sources),
        'bleu': round(bleu, 4),
        'rougeL': round(rougeL, 4),
        'human_review': ''
    })
    time.sleep(0.8)

df = pd.DataFrame(rows)
df.to_csv(output_path, index=False)
print(f'Wrote {len(df)} rows to {output_path}')
df[['question', 'bleu', 'rougeL']].head(20)

In [ ]:
# Summary statistics
in_syllabus = df[df['generated'] != "I'm focused on Class 8 Science; try re-phrasing your question."]
out_of_syllabus = df[df['generated'] == "I'm focused on Class 8 Science; try re-phrasing your question."]

print(f'In-syllabus questions answered : {len(in_syllabus)}')
print(f'Out-of-syllabus (fallback)     : {len(out_of_syllabus)}')
print()
print('Mean BLEU   (in-syllabus):', round(in_syllabus['bleu'].mean(), 4))
print('Mean ROUGE-L (in-syllabus):', round(in_syllabus['rougeL'].mean(), 4))